In [ ]:
from functools import partial

import jax
import jax.numpy as jnp
import netket as nk
from flax import nnx
from netket.experimental.operator import FermiHubbardJax

**Setup:**
Consider a $L \times L$ square lattice in momentum space. The Hubbard Hamiltonian reads:

$\newcommand{\bsl}[1]{\boldsymbol{#1}}$
\begin{align}
\mathcal{H}_0 &= \sum_{\bsl{k}, \sigma} \varepsilon(\bsl{k}) c*{\bsl{k}\sigma}^{\dagger}c*{\bsl{k}\sigma} \\
\mathcal{H}_{\text{int}} &= \frac{U}{L^2} \sum_{\bsl{k}, \bsl{k'}, \bsl{q}} c*{\bsl{k}+\bsl{q}, \uparrow}^{\dagger} c*{\bsl{k}, \uparrow} c*{\bsl{k'}-\bsl{q}, \downarrow}^{\dagger} c*{\bsl{k'}, \downarrow} \\
\mathcal{H} &= \mathcal{H}_0 + \mathcal{H}_{\text{int}},
\end{align}

where $\varepsilon(\bsl{k}) = -2t\left(\cos (k_x) + \cos(k_y)\right)$, $U$ is the on-site repulsion, and $t$ is the hopping parameter. If we convert this Hamiltonian to real space (which is what our NN expects for input), we get:

$$\mathcal{H}=-t\sum_{\langle{\bsl{i}}, \bsl{j}\rangle{},\sigma} \left(c_{\bsl{i}\sigma}^{\dagger} c_{\bsl{j}\sigma} + c_{\bsl{j}\sigma}^{\dagger} c_{\bsl{i}\sigma}\right) + U\sum_{\bsl{i}}n_{\bsl{i}\uparrow}n_{\bsl{i}\downarrow}$$

**Goal:**
We need to minimize
$$E = \frac{\langle{\Psi}|\mathcal{H}|\Psi\rangle{}}{\langle{\Psi}|\Psi\rangle{}}$$


In [ ]:
# Model parameters
L = 3
n = 4
diff = 0
t = 1.0
U = 2.0

# diff = n_spin_up - n_spin_down
# diff and n should have the same parity
n_spin_up = (n + diff) // 2
n_spin_down = (n - diff) // 2

In [ ]:
# make the lattice
lattice = nk.graph.Square(L, pbc=True)
num_lattice_sites = lattice.n_nodes
print(f"{num_lattice_sites = }")

In [ ]:
# make the Hilbert Space
hi = nk.hilbert.SpinOrbitalFermions(
    num_lattice_sites, s=1 / 2, n_fermions_per_spin=(n_spin_down, n_spin_up)
)

In [ ]:
# 1. Generate a random batch of 5 configurations directly from the Hilbert space
key = jax.random.PRNGKey(42)
mock_x = hi.random_state(key, size=5)

print(f"Shape of mock_x from Hilbert space: {mock_x.shape}")
print(f"What mock_x looks like:\n{mock_x}")

In [ ]:
# define the Hamiltonian
H = FermiHubbardJax(hilbert=hi, t=t, U=U, graph=lattice)
print(f"{H.max_conn_size=}")

In [ ]:
# now implement the NN ansatz

# generic MLP helper class
class MLP(nnx.Module):
    def __init__(
        self,
        hidden_layers: int,
        input_dim: int,
        dim_feedforward: int,
        output_dim: int,
        rngs: nnx.Rngs,
        activation=nnx.gelu,
    ):
        # make it
        net = [
            nnx.Linear(input_dim, dim_feedforward, rngs=rngs),
            activation,
        ]
        for _ in range(hidden_layers):
            net.append(nnx.Linear(dim_feedforward, dim_feedforward, rngs=rngs))
            net.append(activation)
        net.append(nnx.Linear(dim_feedforward, output_dim, rngs=rngs))
        self.MLP = nnx.Sequential(*net)

    def __call__(self, x: jax.Array):
        return self.MLP(x)


# quick helper function
def view_as_complex(x: jax.Array):
    return x[..., 0] + (1j * x[..., 1])


class log_psi_2D_spinful(nnx.Module):
    def __init__(
        self,
        L: int,
        n: int,
        hidden_layers: int,
        dim_feedforward: int,
        rngs: nnx.Rngs,
        activation=nnx.gelu,
    ):
        self.L = L
        self.n = n
        # weights, drawn from a Gaussian distribution (for now)
        self.w_s = nnx.Param(rngs.normal((2 * (L**2),)))
        # instantiate MLPs
        self.f = MLP(hidden_layers, 5, dim_feedforward, 2, rngs, activation)
        self.g_1_prime = MLP(hidden_layers, 1, dim_feedforward, 2, rngs, activation)
        self.g_2_prime = MLP(hidden_layers, 1, dim_feedforward, 2, rngs, activation)

    # antisymmetric factor $F: \widetilde{\Lambda}^n \rightarrow \mathbb{C}$
    def F_antisymmetric(self, occ_num: jax.Array):
        # occ_num should be of shape (batch_size, 2*L^2) in an occupation-number basis
        _, nonzero_indices = jax.lax.top_k(occ_num, k=self.n, axis=-1)
        y_indices = nonzero_indices % self.L
        temp = nonzero_indices // self.L
        x_indices = temp % self.L
        spin_indices = temp // self.L
        # now we have x, y, and \sigma
        # convert spatial coords to complex numbers e^{2\pi i x/L}
        z_x = jnp.stack(
            [
                jnp.cos((2 * jnp.pi * x_indices) / self.L),
                jnp.sin((2 * jnp.pi * x_indices) / self.L),
                jnp.cos((2 * jnp.pi * y_indices) / self.L),
                jnp.sin((2 * jnp.pi * y_indices) / self.L),
                (2 * spin_indices) - 1,  # encode spin as ±1
            ],
            axis=-1,
        )  # should be (batch_size, n, 5) now
        # pass through f MLP
        f_x = view_as_complex(self.f(z_x))  # complex (batch_size, n)
        vandermonde_matrix = jax.vmap(partial(jnp.vander, increasing=True))(
            f_x
        )  # should be of shape (batch_size, n, n)
        # take the determinant
        sign, logabsdet = jnp.linalg.slogdet(vandermonde_matrix)
        return (
            sign,  # complex
            logabsdet,  # real
        )  # both should be vectors of shape (batch_size)

    def __call__(self, occ_num: jax.Array):
        # occ_num should be of shape (batch_size, 2*L^2) in an occupation-number basis
        # compute the antisymmetric function F
        sign, logabsdet = self.F_antisymmetric(
            occ_num
        )  # complex/real vectors of shape (batch_size)
        # compute the symmetric function g
        # first calculate eta
        # take matrix-vector product with w_s
        eta = jnp.expand_dims(
            jnp.matmul(occ_num, self.w_s), axis=-1
        )  # shape (batch_size, 1), dtype=float
        # now combine them together via:
        # \Psi = F_1 g_1 + F_2 g_2
        g_1 = view_as_complex(self.g_1_prime(eta))  # shape (batch_size), dtype=complex
        g_2 = view_as_complex(self.g_2_prime(eta))
        # natural log for the final result
        log_psi = logabsdet + jnp.log(
            (sign.real * g_1) + (sign.imag * g_2)
        )  # shape (batch_size), dtype=complex
        return log_psi

In [ ]:
# Instantiate the NN model
model = log_psi_2D_spinful(L, n, hidden_layers=3, dim_feedforward=128, rngs=nnx.Rngs(0))
print(model)

In [ ]:
# Define a Metropolis exchange sampler
exchange_graph = nk.graph.disjoint_union(lattice, lattice)
print(f"Exchange graph size: {exchange_graph.n_nodes}")
sa = nk.sampler.MetropolisExchange(hi, graph=exchange_graph)
print(sa)

In [ ]:
# Define an optimizer
op = nk.optimizer.Sgd(learning_rate=0.01)

# Create a variational state
vstate = nk.vqs.MCState(sa, model, n_samples=512, n_discard_per_chain=100)

# Create a Variational Monte Carlo with SR driver
gs = nk.driver.VMC_SR(
    H,
    op,
    variational_state=vstate,
    diag_shift=0.1,
    linear_solver=nk.optimizer.solver.pinv_smooth,
    mode="complex",
)

# Construct the logger to visualize the data later on
NN_log = nk.logging.RuntimeLog()

# Run the optimization for _ iterations
gs.run(n_iter=500, out=NN_log)